In [13]:
import os
print(os.getcwd())


d:\Projects\fyp-2\src\models


In [10]:
import sys
sys.path.append("..\\..")


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [14]:
import os

PROJECT_ROOT = r"D:\Projects\fyp-2"
os.chdir(PROJECT_ROOT)

print("Working directory set to:", os.getcwd())


Working directory set to: D:\Projects\fyp-2


In [11]:
from src.models.fusion_dataset import FusionDataset
from src.models.baseline_classifier import BaselineClassifier


In [15]:
train_dataset = FusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="train",
    label_map={"dementia": 1, "no_dementia": 0}
)

val_dataset = FusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="val",
    label_map={"dementia": 1, "no_dementia": 0}
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 249
Val samples: 54


In [16]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)


In [17]:
model = BaselineClassifier(input_dim=1536).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model


BaselineClassifier(
  (net): Sequential(
    (0): Linear(in_features=1536, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [18]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE).float()

            logits = model(x)
            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return acc, f1, all_preds, all_labels


In [19]:
EPOCHS = 20

history = {
    "train_loss": [],
    "val_acc": [],
    "val_f1": []
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE).float()

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    val_acc, val_f1, _, _ = evaluate(model, val_loader)

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.3f} | "
        f"Val F1: {val_f1:.3f}"
    )


d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

Epoch 01 | Train Loss: 0.6568 | Val Acc: 0.574 | Val F1: 0.000


d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

Epoch 02 | Train Loss: 0.6331 | Val Acc: 0.574 | Val F1: 0.000


d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

...

Epoch 18 | Train Loss: 0.3531 | Val Acc: 0.722 | Val F1: 0.516


d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

Epoch 19 | Train Loss: 0.3338 | Val Acc: 0.685 | Val F1: 0.414


d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

Epoch 20 | Train Loss: 0.3098 | Val Acc: 0.722 | Val F1: 0.545


In [20]:
best_epoch = np.argmax(history["val_f1"])
print("Best Epoch:", best_epoch + 1)
print("Best Val Accuracy:", history["val_acc"][best_epoch])
print("Best Val F1:", history["val_f1"][best_epoch])


Best Epoch: 20
Best Val Accuracy: 0.7222222222222222
Best Val F1: 0.5454545454545454


In [21]:
_, _, preds, labels = evaluate(model, val_loader)

print(classification_report(
    labels,
    preds,
    target_names=["No Dementia", "Dementia"]
))


              precision    recall  f1-score   support

 No Dementia       0.68      0.97      0.80        31
    Dementia       0.90      0.39      0.55        23

    accuracy                           0.72        54
   macro avg       0.79      0.68      0.67        54
weighted avg       0.77      0.72      0.69        54



d:\Projects\fyp-2\src\models\..\..\src\models\fusion_dataset.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  wavlm_feat = torch.load(wavlm_path)      # [T, 768]
d:\Proj

In [22]:
torch.save(model.state_dict(), "baseline_fusion_model.pt")
print("Baseline model saved.")


Baseline model saved.
